<a href="https://colab.research.google.com/github/SbouriAbdelmajid/MINI_PROJECT/blob/main/codeAP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

import pandas as pd
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
gender_submission = pd.read_csv("gender_submission.csv")

print("✅ Fichiers chargés avec succès !")


Importation de la fonction d’upload :

from google.colab import files permet d’utiliser la fonction d’upload de Colab pour importer des fichiers depuis ton ordinateur.

Upload des fichiers :

uploaded = files.upload() ouvre une boîte de dialogue dans le navigateur pour sélectionner les fichiers CSV nécessaires (train.csv, test.csv, gender_submission.csv).

Les fichiers sont ensuite stockés dans le répertoire /content/ de Colab.

Chargement des fichiers avec pandas :

pd.read_csv("train.csv") lit le fichier d’entraînement et le stocke dans un DataFrame nommé train.

pd.read_csv("test.csv") lit le fichier de test et le stocke dans test.

pd.read_csv("gender_submission.csv") lit le fichier d’exemple de soumission Kaggle et le stocke dans gender_submission.

Confirmation :

print("✅ Fichiers chargés avec succès !") affiche un message confirmant que les fichiers ont été correctement importés et sont prêts à être utilisés.

In [ ]:
# 1. Import des librairies
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score


Importation des librairies principales :

pandas : manipulation des données tabulaires (CSV → DataFrame).

numpy : calculs numériques et manipulation de tableaux.

Préparation des données :

train_test_split : séparer les données en ensembles d’entraînement et de test.

cross_val_score : effectuer une validation croisée pour évaluer la robustesse des modèles.

LabelEncoder : transformer les variables catégorielles (ex. Sexe, Port d’embarquement) en valeurs numériques.

StandardScaler : normaliser les variables numériques pour éviter les biais liés aux échelles différentes.

Algorithmes de classification utilisés :

LogisticRegression : modèle probabiliste pour classification binaire.

DecisionTreeClassifier : arbre de décision basé sur des règles.

KNeighborsClassifier : classification par proximité des voisins.

RandomForestClassifier : ensemble d’arbres de décision pour améliorer la performance et réduire le surapprentissage.

Évaluation des modèles :

accuracy_score : mesure la proportion de prédictions correctes (métrique de base pour comparer les modèles).

In [ ]:
# 2. Charger les fichiers
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
gender_submission = pd.read_csv("gender_submission.csv")


Chargement du fichier train.csv :

Contient les données d’entraînement avec la variable cible Survived.

Sert à entraîner et valider les modèles de Machine Learning.

Chargement du fichier test.csv :

Contient les données des passagers pour lesquels on doit prédire la survie.

Ne contient pas la colonne Survived.

Chargement du fichier gender_submission.csv :

Exemple de soumission fourni par Kaggle.

Sert de modèle pour formater correctement le fichier final submission.csv.

In [ ]:
train["Age"].fillna(train["Age"].median(), inplace=True)
test["Age"].fillna(test["Age"].median(), inplace=True)
train["Embarked"].fillna(train["Embarked"].mode()[0], inplace=True)
test["Fare"].fillna(test["Fare"].median(), inplace=True)

Remplissage des valeurs manquantes pour l’âge (train et test) :

fillna(train["Age"].median(), inplace=True) remplace les valeurs manquantes de la colonne Age par la médiane des âges.

La médiane est choisie car elle est robuste aux valeurs extrêmes (outliers).

On applique la même logique au dataset de test.

Remplissage des valeurs manquantes pour le port d’embarquement (train) :

fillna(train["Embarked"].mode()[0], inplace=True) remplace les valeurs manquantes de la colonne Embarked par la valeur la plus fréquente (mode).

Cela évite de perdre des lignes et conserve la cohérence des données.

Remplissage des valeurs manquantes pour le tarif (test) :

fillna(test["Fare"].median(), inplace=True) remplace les valeurs manquantes de la colonne Fare par la médiane des tarifs.

Cela permet de garder une distribution réaliste des prix sans être influencé par des valeurs extrêmes.

In [ ]:
for col in ["Sex", "Embarked"]:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col])
    test[col] = le.transform(test[col])

Boucle sur les colonnes catégorielles :

La boucle for col in ["Sex", "Embarked"]: parcourt les deux colonnes catégorielles du dataset (Sex et Embarked) qui doivent être converties en valeurs numériques.

Création d’un encodeur :

le = LabelEncoder() initialise un objet de la classe LabelEncoder de scikit-learn.

Cet encodeur transforme des catégories textuelles (ex. "male", "female", "S", "C", "Q") en entiers (0, 1, 2…).

Encodage des colonnes du dataset d’entraînement :

train[col] = le.fit_transform(train[col]) applique l’encodage sur la colonne correspondante du dataset d’entraînement.

La méthode fit_transform() apprend les catégories uniques et les convertit en valeurs numériques.

Transformation des colonnes du dataset de test :

test[col] = le.transform(test[col]) applique le même encodage au dataset de test.

Cela garantit que les catégories sont codées de manière identique dans les deux datasets (train et test).

In [ ]:
features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]
X = train[features]
y = train["Survived"]
X_test_final = test[features]

Définition des variables explicatives (features) :

La liste features contient les colonnes retenues pour l’entraînement du modèle :

Pclass (classe sociale du passager)

Sex (sexe encodé en numérique)

Age (âge du passager)

SibSp (nombre de frères/sœurs ou conjoint à bord)

Parch (nombre de parents/enfants à bord)

Fare (prix du billet)

Embarked (port d’embarquement encodé en numérique)

Création de la matrice des variables explicatives (X) :

X = train[features] sélectionne uniquement les colonnes définies dans features à partir du dataset d’entraînement.

C’est la matrice des données d’entrée pour les modèles.

Définition de la variable cible (y) :

y = train["Survived"] extrait la colonne Survived du dataset d’entraînement.

C’est la variable que l’on cherche à prédire (0 = non survécu, 1 = survécu).

Préparation du dataset de test (X_test_final) :

X_test_final = test[features] sélectionne les mêmes colonnes dans le dataset de test.

Cela garantit que les modèles seront appliqués sur des données ayant la même structure que celles utilisées pour l’entraînement.

In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(X)
X_test_final = scaler.transform(X_test_final)

Initialisation du normaliseur :

scaler = StandardScaler() crée un objet de la classe StandardScaler de scikit-learn.

Cet outil permet de normaliser les variables numériques en leur donnant une moyenne de 0 et un écart-type de 1.

Normalisation des données d’entraînement :

X = scaler.fit_transform(X) ajuste le normaliseur (fit) sur les données d’entraînement puis applique la transformation (transform).

Chaque variable est centrée et réduite, ce qui évite que certaines variables (comme Fare) dominent les autres à cause de leur échelle.

Normalisation des données de test :

X_test_final = scaler.transform(X_test_final) applique la même transformation aux données de test.

On utilise uniquement transform (et non fit_transform) pour garantir que les données de test sont normalisées selon les mêmes paramètres que l’entraînement.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

Séparation des données en deux ensembles :

La fonction train_test_split divise les données en un ensemble d’entraînement et un ensemble de validation.

Cela permet de tester les modèles sur des données qu’ils n’ont pas vues pendant l’entraînement.

Variables créées :

X_train : les variables explicatives utilisées pour entraîner le modèle.

X_val : les variables explicatives utilisées pour valider le modèle.

y_train : la cible (survie ou non) associée à X_train.

y_val : la cible associée à X_val.

Paramètres utilisés :

test_size=0.2 → 20 % des données sont réservées pour la validation, 80 % pour l’entraînement.

random_state=42 → fixe la graine aléatoire pour garantir que la séparation est reproductible (même découpage à chaque exécution).

In [ ]:
models = {
    "Régression logistique": LogisticRegression(max_iter=1000),
    "Arbre de décision": DecisionTreeClassifier(max_depth=5),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

Création d’un dictionnaire de modèles :

La variable models est un dictionnaire où chaque clé est le nom d’un algorithme et chaque valeur est l’objet du modèle correspondant.

Cela permet de stocker plusieurs modèles dans une seule structure et de les parcourir facilement pour les entraîner et les comparer.

Régression logistique :

LogisticRegression(max_iter=1000) initialise un modèle de régression logistique.

Le paramètre max_iter=1000 augmente le nombre d’itérations pour assurer la convergence de l’algorithme.

Arbre de décision :

DecisionTreeClassifier(max_depth=5) crée un arbre de décision limité à une profondeur de 5.

Cela évite le surapprentissage (overfitting) en limitant la complexité du modèle.

KNN (K-Nearest Neighbors) :

KNeighborsClassifier(n_neighbors=5) définit un modèle KNN avec 5 voisins.

La prédiction se fait en fonction des 5 passagers les plus proches dans l’espace des variables.

Random Forest :

RandomForestClassifier(n_estimators=100, random_state=42) crée une forêt aléatoire composée de 100 arbres de décision.

Le paramètre random_state=42 garantit la reproductibilité des résultats.

Ce modèle est souvent plus robuste car il combine plusieurs arbres pour réduire la variance.

In [ ]:
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    acc = accuracy_score(y_val, y_pred)
    print(f"{name} -> Accuracy: {acc:.4f}")

Boucle sur les modèles :

for name, model in models.items(): parcourt le dictionnaire models.

À chaque itération, name contient le nom du modèle (ex. "Régression logistique") et model l’objet du modèle correspondant.

Entraînement du modèle :

model.fit(X_train, y_train) entraîne le modèle sur les données d’entraînement (X_train, y_train).

Chaque algorithme apprend à prédire la survie des passagers à partir des variables explicatives.

Prédiction sur l’ensemble de validation :

y_pred = model.predict(X_val) génère les prédictions du modèle sur les données de validation.

Cela permet de tester le modèle sur des données qu’il n’a pas vues pendant l’entraînement.

Calcul de l’accuracy :

acc = accuracy_score(y_val, y_pred) calcule la proportion de prédictions correctes.

L’accuracy est une métrique simple mais utile pour comparer les modèles.

Affichage des résultats :

print(f"{name} -> Accuracy: {acc:.4f}") affiche le nom du modèle et son score d’accuracy avec 4 décimales.

Cela permet de comparer rapidement les performances des différents algorithmes.

In [ ]:
best_model = RandomForestClassifier(n_estimators=100, random_state=42)
best_model.fit(X, y)
predictions = best_model.predict(X_test_final)

Définition du meilleur modèle :

best_model = RandomForestClassifier(n_estimators=100, random_state=42) crée un modèle de type Random Forest.

n_estimators=100 → la forêt est composée de 100 arbres de décision.

random_state=42 → fixe la graine aléatoire pour assurer la reproductibilité des résultats.

Ce modèle est choisi car il a montré les meilleures performances lors de la phase de validation.

Entraînement du modèle sur toutes les données :

best_model.fit(X, y) entraîne le modèle sur l’ensemble complet des données d’entraînement (X, y).

Cela permet d’utiliser toutes les informations disponibles pour améliorer la précision avant de prédire sur le dataset de test.

Prédictions sur le dataset de test :

predictions = best_model.predict(X_test_final) applique le modèle entraîné aux données du fichier test.csv.

Le résultat est un tableau de valeurs binaires (0 ou 1) indiquant si chaque passager du test a survécu ou non.

In [ ]:
submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": predictions
})
submission.to_csv("submission.csv", index=False)

print("✅ Fichier submission.csv généré avec succès !")

Création du DataFrame de soumission :

pd.DataFrame({...}) construit un nouveau tableau avec deux colonnes :

PassengerId → identifiant unique de chaque passager du dataset de test.

Survived → prédictions générées par le modèle (0 = non survécu, 1 = survécu).

Ce format correspond exactement à celui exigé par Kaggle.

Exportation du fichier CSV :

submission.to_csv("submission.csv", index=False) enregistre le DataFrame dans un fichier CSV nommé submission.csv.

L’option index=False évite d’ajouter une colonne d’index inutile dans le fichier.

Ce fichier est prêt à être soumis sur Kaggle pour évaluation.

Confirmation visuelle :

print("✅ Fichier submission.csv généré avec succès !") affiche un message confirmant que le fichier a été créé correctement.

Cela permet de vérifier que l’étape finale s’est déroulée sans erreur.